# Implementasi Bab 2 — Workflow, CRISP-DM, dan Bias-Variance

Notebook membuat learning curve untuk mendiagnosis underfitting, seimbang, dan overfitting serta memetakan eksperimen ke enam fase CRISP-DM [ML-MODUL-01, hlm. 20–21].


In [ ]:
import random
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import learning_curve, train_test_split
from sklearn.tree import DecisionTreeClassifier

SEED = 42
random.seed(SEED)
np.random.seed(SEED)


In [ ]:
X, y = make_classification(
    n_samples=1000,
    n_features=10,
    n_informative=6,
    weights=[0.8, 0.2],
    random_state=SEED,
)
print("Distribusi kelas:", np.bincount(y))


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
print("Proporsi train:", np.bincount(y_train) / len(y_train))
print("Proporsi test :", np.bincount(y_test) / len(y_test))


In [ ]:
model = DecisionTreeClassifier(max_depth=4, random_state=SEED)
train_sizes, train_scores, validation_scores = learning_curve(
    model,
    X_train,
    y_train,
    cv=5,
    train_sizes=np.linspace(0.1, 1.0, 8),
    scoring="accuracy",
    n_jobs=-1,
)
train_mean = train_scores.mean(axis=1)
validation_mean = validation_scores.mean(axis=1)
print("Train terbesar:", round(train_mean[-1], 3))
print("Validasi terbesar:", round(validation_mean[-1], 3))
print("Gap:", round(train_mean[-1] - validation_mean[-1], 3))


In [ ]:
plt.plot(train_sizes, train_mean, "o-", label="Training Score")
plt.plot(train_sizes, validation_mean, "o-", label="Validation Score")
plt.xlabel("Jumlah Data Latih")
plt.ylabel("Akurasi")
plt.title("Learning Curve — max_depth=4")
plt.legend()
plt.show()


## Latihan 1: pemetaan CRISP-DM

Contoh studi kasus churn pelanggan:

1. **Business Understanding:** tentukan biaya pelanggan pergi dan target retensi.
2. **Data Understanding:** audit distribusi churn, kualitas, dan potensi bias.
3. **Data Preparation:** split, imputasi, encoding, dan pencegahan leakage.
4. **Modeling:** bandingkan baseline dan beberapa tingkat kompleksitas.
5. **Evaluation:** gunakan recall/F1 dan validasi lintas fold.
6. **Deployment:** integrasikan skor risiko dan monitor drift.

Pemetaan ini adalah jawaban contoh untuk latihan konseptual [ML-MODUL-01, hlm. 21].


In [ ]:
depth_results = []
for depth in [1, 4, None]:
    candidate = DecisionTreeClassifier(max_depth=depth, random_state=SEED)
    _, train_scores_i, validation_scores_i = learning_curve(
        candidate,
        X_train,
        y_train,
        cv=5,
        train_sizes=np.linspace(0.1, 1.0, 6),
        scoring="accuracy",
        n_jobs=-1,
    )
    train_final = train_scores_i.mean(axis=1)[-1]
    validation_final = validation_scores_i.mean(axis=1)[-1]
    gap = train_final - validation_final
    diagnosis = "overfitting" if gap > 0.08 else ("underfitting" if validation_final < 0.75 else "seimbang")
    depth_results.append((str(depth), train_final, validation_final, diagnosis))

display(pd.DataFrame(depth_results, columns=["max_depth", "train", "validation", "diagnosis"]))


## Interpretasi

Gap train–validation yang kecil pada skor cukup tinggi menunjukkan keseimbangan bias-variance yang lebih baik. Diagnosis tetap perlu dilihat bersama konteks biaya kesalahan dan test set yang tidak ikut memilih model [ML-MODUL-01, hlm. 21].
